# Optimizers — Exercises

Hands-on exercises for `04_advanced_topics/08_optimizers`. Everything uses the NumPy code in `implementation.py` and synthetic data only.

Work through the cells in order. Cells marked **TODO** are for you to complete.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np

# Works locally and in Colab (clones the repo if needed)
MODULE = "04_advanced_topics/08_optimizers"
if not os.path.exists("implementation.py"):
    if not os.path.exists("ml-algo"):
        os.system("git clone -q https://github.com/imam-db/ml-algo.git")
    os.chdir(os.path.join("ml-algo", MODULE))
sys.path.insert(0, os.getcwd())

from implementation import (
    OPTIMIZERS,
    SGD,
    Adam,
    AdamW,
    MLPModel,
    accuracy,
    cosine_decay,
    make_moons_np,
    make_quadratic,
    numerical_gradient,
    optimize,
    plot_trajectories,
    rosenbrock,
    step_decay,
    train,
    warmup,
)

%matplotlib inline

## Exercise 1 — The stability limit of SGD

For $f(x) = \tfrac12 (x_1^2 + 50 x_2^2)$ the largest curvature is $\lambda_{\max} = 50$, so theory says SGD is stable only for $\eta < 2/50 = 0.04$.

**TODO:** run SGD for 200 steps with learning rates around 0.04 and find the largest one that still converges.

In [ ]:
quad = make_quadratic(50.0)
for lr in [0.01, 0.03, 0.039, 0.041, 0.05]:
    res = optimize(quad, [5.0, 1.0], SGD(lr=lr), n_steps=200)
    print(f"lr={lr:<6} final f = {res['values'][-1]:.3e}")

## Exercise 2 — Rotate the valley

The benchmark quadratic is axis-aligned, which favours per-coordinate methods (AdaGrad, RMSProp, Adam).

**TODO:** complete `rotated_quadratic` so that $f(x) = \tfrac12 x^\top R^\top D R x$ with a 45° rotation $R$, then compare steps-to-tolerance for SGD, Momentum, RMSProp and Adam.

In [ ]:
def rotated_quadratic(condition=50.0, angle=np.pi / 4):
    c, s = np.cos(angle), np.sin(angle)
    R = np.array([[c, -s], [s, c]])
    A = R.T @ np.diag([1.0, condition]) @ R
    def f(x):
        x = np.asarray(x, dtype=float)
        # TODO: return value and gradient
        return 0.5 * x @ A @ x, A @ x
    return f

rq = rotated_quadratic()
for name, lr in [("SGD", 0.03), ("Momentum", 0.01), ("RMSProp", 0.1), ("Adam", 0.3)]:
    res = optimize(rq, [5.0, 1.0], OPTIMIZERS[name](lr=lr), n_steps=5000, tol=1e-6)
    print(name, res["converged_at"])

## Exercise 3 — Adam's bias correction

With bias correction the first Adam step moves every parameter by (almost exactly) `lr`, whatever the gradient scale.

**TODO:** confirm this, then switch `bias_correction=False` and measure how much bigger the first step becomes. Compare with $(1-\beta_1)/\sqrt{1-\beta_2}$.

In [ ]:
for bc in [True, False]:
    p = np.zeros(3)
    Adam(lr=0.01, bias_correction=bc).step([p], [np.array([1e-4, 1.0, 1e4])])
    print("bias_correction =", bc, "first step =", -p)
print("theory ratio:", 0.1 / np.sqrt(0.001))

## Exercise 4 — Coupled L2 vs decoupled weight decay

**TODO:** train the MLP with `Adam(weight_decay=1e-2)` and `AdamW(weight_decay=1e-2)` and compare the final weight norms and accuracy. Which one shrinks the weights more evenly?

In [ ]:
X, y = make_moons_np(400, seed=0)
for name, opt in [("Adam + L2", Adam(lr=0.01, weight_decay=1e-2)), ("AdamW", AdamW(lr=0.01, weight_decay=1e-2))]:
    m = MLPModel(2, hidden=16, seed=0)
    train(m, X, y, opt, epochs=60, batch_size=32)
    norm = np.sqrt(sum((p**2).sum() for p in m.params))
    print(f"{name:10s} acc={accuracy(m, X, y):.3f}  ||theta||={norm:.2f}")

## Exercise 5 — Learning-rate schedules

**TODO:** plot the three schedules below over 1000 steps, then train the MLP with `SGD(lr=schedule)` for each and compare final loss.

In [ ]:
T = 1000
schedules = {
    "step": step_decay(0.1, step_size=300, gamma=0.3),
    "cosine": cosine_decay(0.1, total_steps=T),
    "warmup+cosine": warmup(cosine_decay(0.1, total_steps=T - 100), warmup_steps=100),
}
for name, s in schedules.items():
    plt.plot([s(t) for t in range(T)], label=name)
plt.xlabel("step"); plt.ylabel("learning rate"); plt.legend(); plt.show()

## Exercise 6 — Trajectories

Visualise how optimizers move on the Rosenbrock function. **TODO:** try different learning rates and `beta` values and describe what you see.

In [ ]:
plot_trajectories(rosenbrock, [-1.5, 2.0],
                  {"SGD": {"lr": 0.001}, "Momentum": {"lr": 0.001}, "Adam": {"lr": 0.1}},
                  n_steps=1500)
plt.show()

## Exercise 7 — Build your own optimizer

**TODO:** implement AMSGrad (Reddi et al., 2018): like Adam, but keep $\hat v_t^{\max} = \max(\hat v_{t-1}^{\max}, v_t)$ and divide by its square root. Subclass `Adam`, then add it to the benchmark.

In [ ]:
class AMSGrad(Adam):
    name = "AMSGrad"
    def _update(self, i, p, g, lr):
        # TODO: your code here
        raise NotImplementedError

# res = optimize(rosenbrock, [-1.5, 2.0], AMSGrad(lr=0.1), n_steps=5000, tol=1e-6)
# print(res["converged_at"])

## Checking gradients

Before trusting any optimizer, check your gradients. `numerical_gradient` uses central differences.

In [ ]:
x = np.array([-1.2, 0.8])
print(rosenbrock(x)[1], numerical_gradient(lambda z: rosenbrock(z)[0], x))